In [1]:
import torch
import random
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Tekrarlanabilirlik için tohumları sabitleyelim (Random Seed)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print("1. Veri Seti İndiriliyor (AG News)...")
# HuggingFace'den AG News veri setini çekiyoruz (Otomatik iner)
dataset = load_dataset("ag_news")

# Veri yapısını görelim
print(f"Eğitim verisi sayısı: {len(dataset['train'])}")
print(f"Test verisi sayısı: {len(dataset['test'])}")
print("Örnek Veri:", dataset['train'][0])

# Etiketlerin anlamları: {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
label_map = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

# Eğitim ve Test verilerini ayıralım (Metin ve Etiketler)
# Hızlanmak için eğitim setinden sadece 5.000, testten 1.000 örnek alalım (Local PC için)
train_subset = dataset['train'].shuffle(seed=42).select(range(5000))
test_subset = dataset['test'].shuffle(seed=42).select(range(1000))

X_train_text = train_subset['text']
y_train = train_subset['label']

X_test_text = test_subset['text']
y_test = test_subset['label']

print("\n2. Baseline Model İçin Veri Vektörleştiriliyor (Bag-of-Words)...")
# Metindeki gibi CountVectorizer kullanıyoruz
vectorizer = CountVectorizer(max_features=10_000, binary=True)
X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)

print("3. Lojistik Regresyon Modeli Eğitiliyor...")
# Basit model eğitimi
model_baseline = LogisticRegression(random_state=42, max_iter=1000)
model_baseline.fit(X_train_vec, y_train)

# Tahmin ve Skor
y_train_pred = model_baseline.predict(X_train_vec)
y_test_pred = model_baseline.predict(X_test_vec)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"\n--- SONUÇLAR ---")
print(f"Baseline Eğitim Doğruluğu: %{train_acc * 100:.2f}")
print(f"Baseline Test Doğruluğu:   %{test_acc * 100:.2f}")
print("Bu skor, GPT-2 ile geçmemiz gereken minimum hedeftir.")

/opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Veri Seti İndiriliyor (AG News)...
Eğitim verisi sayısı: 120000
Test verisi sayısı: 7600
Örnek Veri: {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

2. Baseline Model İçin Veri Vektörleştiriliyor (Bag-of-Words)...
3. Lojistik Regresyon Modeli Eğitiliyor...

--- SONUÇLAR ---
Baseline Eğitim Doğruluğu: %99.98
Baseline Test Doğruluğu:   %85.50
Bu skor, GPT-2 ile geçmemiz gereken minimum hedeftir.


In [2]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Tokenizer ve Model Ayarları
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2'de padding token yoktur, EOS token'ı padding olarak atıyoruz (Metindeki gibi)
tokenizer.pad_token = tokenizer.eos_token

# Etiketleri Kelimeye Çevirme Haritası
# AG News etiketleri: 0: World, 1: Sports, 2: Business, 3: Sci/Tech
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# 2. Özel Dataset Sınıfı
class GPTClassificationDataset(Dataset):
    def __init__(self, data_source, tokenizer, max_length=128):
        self.data = data_source
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        label_word = id2label[item['label']]

        # FORMATLAMA (Metindeki Table 5.1 mantığı)
        # Soru kısmı (Prompt)
        prompt = f"Predict Category: {text}\nCategory:"
        # Cevap kısmı (Completion)
        completion = f" {label_word}" 
        
        # Tam metin (Eğitim için)
        full_text = prompt + completion + tokenizer.eos_token

        # Tokenize İşlemi
        # Tam metni tokenize et
        full_encoding = tokenizer(
            full_text, 
            max_length=self.max_length, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        
        input_ids = full_encoding["input_ids"][0]
        attention_mask = full_encoding["attention_mask"][0]
        
        # LABELS OLUŞTURMA (Loss Masking -100)
        labels = input_ids.clone()
        
        # Prompt kısmının uzunluğunu bulup maskeleyeceğiz
        prompt_len = len(tokenizer.encode(prompt))
        
        # Prompt kısmını -100 yap (Model burayı öğrenmesin, sadece cevabı öğrensin)
        # Not: Eğer metin çok uzunsa ve kesildiyse (truncation), prompt_len hata vermemeli.
        safe_prompt_len = min(prompt_len, self.max_length)
        labels[:safe_prompt_len] = -100
        
        # Padding olan kısımları da -100 yap (Model boşlukları öğrenmesin)
        labels[input_ids == tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# 3. Dataset ve DataLoader Oluşturma
print("Veri hazırlanıyor (Tokenizasyon)...")

# Daha önce ayırdığımız train_subset ve test_subset'i kullanıyoruz
train_dataset = GPTClassificationDataset(train_subset, tokenizer)
test_dataset = GPTClassificationDataset(test_subset, tokenizer)

# Batch Size (Metindeki gibi 16 veya bilgisayarın gücüne göre 8)
BATCH_SIZE = 8 

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# Test için shuffle kapalı
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Hazır! Train Batch Sayısı: {len(train_loader)}")

# 4. Kontrol: Modelin ne göreceğine bakalım
sample = train_dataset[0]
print("\n--- ÖRNEK VERİ KONTROLÜ ---")
print("Ham Girdi ID'leri:", sample['input_ids'][:20]) # İlk 20 token
print("Maskelenmiş Etiketler:", sample['labels'][:20]) # İlk 20 etiket
print("\nNot: Etiketlerde '-100' gördüğün yerler sorunun kendisidir (Maskelenmiş).")
print("Sadece pozitif sayı olan yerler (Cevap) için loss hesaplanacak.")

Veri hazırlanıyor (Tokenizasyon)...
Hazır! Train Batch Sayısı: 625

--- ÖRNEK VERİ KONTROLÜ ---
Ham Girdi ID'leri: tensor([   47, 17407, 21743,    25, 19483,  1582, 47557,   416,  8956, 29560,
         7941,   423,  3181,   867, 11684,   290,  4736,   287, 19483,   284])
Maskelenmiş Etiketler: tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100])

Not: Etiketlerde '-100' gördüğün yerler sorunun kendisidir (Maskelenmiş).
Sadece pozitif sayı olan yerler (Cevap) için loss hesaplanacak.


In [4]:
from transformers import AutoModelForCausalLM
from torch.optim import AdamW
from tqdm.auto import tqdm # İlerleme çubuğu için

# 1. Cihaz Ayarı (GPU varsa kullanalım, yoksa CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Eğitim Cihazı: {device}")

# 2. Modeli Yükle ve Cihaza Taşı
print("Model yükleniyor...")
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

# 3. Hiperparametreler (Metindeki önerilere sadık kalıyoruz)
# Epoch: Veri setinin üzerinden kaç kez geçilecek? (3 genelde yeterlidir)
EPOCHS = 3 
# Learning Rate: İnce ayar için düşük bir oran seçilir.
LEARNING_RATE = 5e-5 

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# 4. Eğitim Döngüsü
print(f"\nEğitim Başlıyor ({EPOCHS} Epoch)...")

# Kayıp değerlerini saklamak için
train_losses = []

model.train() # Modeli eğitim moduna al

for epoch in range(EPOCHS):
    # İlerleme çubuğu (Progress Bar)
    loop = tqdm(train_loader, leave=True)
    total_loss = 0
    
    for batch in loop:
        # Verileri cihaza (GPU/CPU) gönder
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # A. Forward Pass (Tahmin)
        outputs = model(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels
        )
        
        loss = outputs.loss
        
        # B. Backward Pass (Öğrenme)
        optimizer.zero_grad() # Eski gradyanları temizle
        loss.backward()       # Hatayı geriye yay
        optimizer.step()      # Ağırlıkları güncelle
        
        # Görselleştirme
        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"--> Epoch {epoch+1} Tamamlandı. Ortalama Kayıp (Loss): {avg_loss:.4f}")

print("\nEğitim Bitti! Model ince ayarlandı.")

Eğitim Cihazı: mps
Model yükleniyor...

Eğitim Başlıyor (3 Epoch)...


Epoch 1: 100%|██████████| 625/625 [04:43<00:00,  2.20it/s, loss=0.244]  


--> Epoch 1 Tamamlandı. Ortalama Kayıp (Loss): 0.2807


Epoch 2: 100%|██████████| 625/625 [03:37<00:00,  2.88it/s, loss=0.516]   


--> Epoch 2 Tamamlandı. Ortalama Kayıp (Loss): 0.1574


Epoch 3: 100%|██████████| 625/625 [03:35<00:00,  2.90it/s, loss=0.0377]  

--> Epoch 3 Tamamlandı. Ortalama Kayıp (Loss): 0.0909

Eğitim Bitti! Model ince ayarlandı.


In [7]:
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm
import torch

# 1. Hazırlıklar
print("Test süreci başlıyor...")
model.eval() # Modeli değerlendirme moduna al

predictions = []
true_labels = []

# Etiket haritaları (String <-> ID)
# id2label = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'} (Daha önce tanımlamıştık)
label2id = {v: k for k, v in id2label.items()}

# 2. Tahmin Döngüsü (Inference Loop)
# Test setindeki 1000 veri üzerinde dönüyoruz
for i in tqdm(range(len(test_subset)), desc="Tahmin Ediliyor"):
    item = test_subset[i]
    text = item['text']
    true_label_id = item['label']
    
    # Prompt Hazırla (Eğitimdeki formatla birebir aynı olmalı)
    prompt = f"Predict Category: {text}\nCategory:"
    
    # Tokenize et ve GPU'ya taşı
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Üretim Yap (Generation)
    with torch.no_grad():
        # max_new_tokens=5 yaptık, bazen boşluk veya özel karakter üretebilir, garanti olsun.
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=5, 
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False # Greedy decoding (En olası cevabı seç)
        )
    
    # Çıktıyı Metne Çevir (Decode)
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Cevabı Ayıkla (Parsing)
    # Model çıktısı şöyle olur: "Predict Category: ... \nCategory: Sports"
    # Biz sadece "Sports" kısmını istiyoruz.
    try:
        answer = generated_text.split("Category:")[-1].strip()
        # Bazen model "Sports." veya "Sports " gibi çıktılar verebilir, temizleyelim.
        answer = answer.split()[0] if answer else "" # İlk kelimeyi al
        # Noktalama işaretlerini temizle
        answer = answer.strip(".,!?;: ")
    except:
        answer = ""

    # Metni ID'ye çevir
    # Eğer model sözlükte olmayan bir şey ürettiyse (Halüsinasyon), -1 veriyoruz.
    pred_id = label2id.get(answer, -1)
    
    predictions.append(pred_id)
    true_labels.append(true_label_id)

# 3. Sonuç Analizi ve Raporlama
print("\n--- ANALİZ TAMAMLANDI ---")

# Halüsinasyon Kontrolü
invalid_count = predictions.count(-1)
total_count = len(predictions)

if invalid_count > 0:
    print(f"Uyarı: Model {invalid_count} adet geçersiz cevap üretti.")
    print("Bu cevaplar 'yanlış' kabul edilerek skora dahil edilecektir.\n")

# Genel Doğruluk (Accuracy)
# -1 olanlar (geçersizler) otomatik olarak yanlış sayılır, bu doğru bir yaklaşımdır.
acc = accuracy_score(true_labels, predictions)

print(f"===============================")
print(f"BASELINE SKORU:      %85.50")
print(f"GPT-2 FINE-TUNED:    %{acc * 100:.2f}")
print(f"===============================")

# Detaylı Sınıflandırma Raporu (Hata veren kısmı düzelttik)
# 'labels' parametresi ile sadece geçerli [0, 1, 2, 3] sınıflarını raporluyoruz.
# -1 olanlar rapora dahil edilmez ama accuracy'yi düşürür.
valid_label_ids = list(id2label.keys())     # [0, 1, 2, 3]
valid_label_names = list(id2label.values()) # ['World', 'Sports', 'Business', 'Sci/Tech']

print("\nDetaylı Sınıflandırma Raporu:")
print(classification_report(
    true_labels, 
    predictions, 
    labels=valid_label_ids, 
    target_names=valid_label_names,
    zero_division=0
))

# 4. Örnek Hatalar
print("\n--- HATALI TAHMİN ÖRNEKLERİ ---")
shown = 0
for true, pred, item in zip(true_labels, predictions, test_subset):
    if true != pred:
        pred_str = id2label.get(pred, "GEÇERSİZ")
        true_str = id2label[true]
        print(f"Metin: {item['text'][:60]}...")
        print(f"Gerçek: {true_str} | Tahmin: {pred_str}\n")
        shown += 1
        if shown >= 3: break

Test süreci başlıyor...


Tahmin Ediliyor: 100%|██████████| 1000/1000 [01:31<00:00, 10.98it/s]


--- ANALİZ TAMAMLANDI ---
Uyarı: Model 181 adet geçersiz cevap üretti.
Bu cevaplar 'yanlış' kabul edilerek skora dahil edilecektir.

BASELINE SKORU:      %85.50
GPT-2 FINE-TUNED:    %73.20

Detaylı Sınıflandırma Raporu:
              precision    recall  f1-score   support

       World       0.96      0.88      0.92       266
      Sports       0.97      0.98      0.97       246
    Business       0.79      0.93      0.85       246
    Sci/Tech       0.80      0.12      0.20       242

   micro avg       0.89      0.73      0.80      1000
   macro avg       0.88      0.73      0.74      1000
weighted avg       0.88      0.73      0.74      1000


--- HATALI TAHMİN ÖRNEKLERİ ---
Metin: REVIEW: 'Half-Life 2' a Tech Masterpiece (AP) AP - It's been...
Gerçek: Sci/Tech | Tahmin: GEÇERSİZ

Metin: China's inflation rate slows sharply but problems remain (AF...
Gerçek: World | Tahmin: Business

Metin: Intel gets off chip speed roller coaster: Technology India: ...
Gerçek: Sci/Tech | Tahmin: 

In [8]:
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm
import torch

print("İyileştirilmiş Test Süreci Başlıyor...")

# Esnek Eşleştirme Fonksiyonu
def map_generation_to_id(gen_text):
    text = gen_text.lower()
    
    # 1. Sci/Tech Varyasyonları (En çok hata burada)
    if "sci" in text or "tech" in text:
        return 3 # Sci/Tech
    # 2. Sports Varyasyonları
    elif "sport" in text or "game" in text or "ball" in text:
        return 1 # Sports
    # 3. Business Varyasyonları
    elif "busin" in text or "econ" in text or "market" in text:
        return 2 # Business
    # 4. World Varyasyonları
    elif "world" in text or "global" in text or "polit" in text:
        return 0 # World
    
    return -1 # Hala bulunamadıysa

predictions = []
true_labels = []

# Model Eval Modu
model.eval()

for i in tqdm(range(len(test_subset)), desc="Akıllı Tahmin"):
    item = test_subset[i]
    text = item['text']
    true_label_id = item['label']
    
    prompt = f"Predict Category: {text}\nCategory:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        # max_new_tokens'ı biraz artıralım (5 -> 10) ki "Science and Tech" derse kesilmesin
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=10, 
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )
    
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Sadece üretilen cevabı al
    try:
        answer_part = generated_text.split("Category:")[-1].strip()
        # İlk satırı al (bazen model alt satıra geçip saçmalayabilir)
        answer_part = answer_part.split('\n')[0]
    except:
        answer_part = ""
    
    # ESNEK EŞLEŞTİRME
    pred_id = map_generation_to_id(answer_part)
    
    # Hala -1 ise, belki model tam olarak "Sci/Tech" yazmıştır ama casing farklıdır
    if pred_id == -1:
        # Son bir şans: Sözlükte var mı diye bak
        # label2id = {'World': 0, ...}
        # Baş harfi büyük yapıp dene
        cleaned = answer_part.strip().capitalize()
        pred_id = label2id.get(cleaned, -1)

    predictions.append(pred_id)
    true_labels.append(true_label_id)

# SONUÇLARI TEKRAR HESAPLA
acc = accuracy_score(true_labels, predictions)
invalid_count = predictions.count(-1)

print(f"\n===============================")
print(f"BASELINE SKORU:      %85.50")
print(f"GPT-2 (ESNEK):       %{acc * 100:.2f}")
print(f"===============================")
print(f"Hala Geçersiz Sayısı: {invalid_count}")

valid_label_ids = [0, 1, 2, 3]
valid_label_names = ['World', 'Sports', 'Business', 'Sci/Tech']

print("\nFinal Raporu:")
print(classification_report(
    true_labels, 
    predictions, 
    labels=valid_label_ids, 
    target_names=valid_label_names,
    zero_division=0
))

# Model ne üretmiş de biz Sci/Tech saymışız? Örnek görelim:
print("\n--- 'Sci/Tech' İÇİN ESNEK EŞLEŞME ÖRNEKLERİ ---")
shown = 0
for i, pred in enumerate(predictions):
    if pred == 3 and true_labels[i] == 3: # Doğru bilinen Sci/Tech
        # Modelin ürettiği ham metni tekrar çekmiyoruz ama mantığı anladık.
        # Burada sadece tahminin 3'e döndüğünü görüyoruz.
        pass

İyileştirilmiş Test Süreci Başlıyor...


Akıllı Tahmin: 100%|██████████| 1000/1000 [02:48<00:00,  5.95it/s]


BASELINE SKORU:      %85.50
GPT-2 (ESNEK):       %80.40
Hala Geçersiz Sayısı: 0

Final Raporu:
              precision    recall  f1-score   support

       World       0.99      0.83      0.90       266
      Sports       0.97      0.94      0.95       246
    Business       0.91      0.47      0.62       246
    Sci/Tech       0.58      0.98      0.73       242

    accuracy                           0.80      1000
   macro avg       0.86      0.80      0.80      1000
weighted avg       0.86      0.80      0.80      1000


--- 'Sci/Tech' İÇİN ESNEK EŞLEŞME ÖRNEKLERİ ---


In [9]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

print("--- PROFESYONEL YÖNTEM: LOGIT BAZLI SINIFLANDIRMA ---")

# 1. Hedef Kelimelerin Token ID'lerini Bulalım
# Eğitimde formatımız: "Category: World" şeklindeydi (Başında boşluk var)
# Bu yüzden modelin " World", " Sports", " Business", " Sci" tokenlarına verdiği puanlara bakacağız.

target_labels = ["World", "Sports", "Business", "Sci/Tech"]
target_token_ids = []

for label in target_labels:
    # " Sci/Tech" gibi kelimelerin ilk tokenını alacağız (Genelde " Sci")
    # GPT-2 tokenizer'ında kelime başı boşluk önemlidir.
    token_id = tokenizer.encode(" " + label)[0] 
    target_token_ids.append(token_id)
    print(f"Hedef: '{label}' -> İzlenen Token ID: {token_id} ({tokenizer.decode([token_id])})")

# 2. Logit Bazlı Tahmin Döngüsü
model.eval()
predictions = []
true_labels = []

# id2label sözlüğünü (0,1,2,3) hedef token listesindeki sıraya (0,1,2,3) eşliyoruz.
# target_token_ids[0] -> World (Label 0)
# target_token_ids[1] -> Sports (Label 1) vb.

for i in tqdm(range(len(test_subset)), desc="Olasılıklar Hesaplanıyor"):
    item = test_subset[i]
    text = item['text']
    true_label_id = item['label']
    
    # Prompt (Eğitimdeki ile aynı)
    prompt = f"Predict Category: {text}\nCategory:"
    
    # Tokenize et
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        # Forward pass (Generation yapmıyoruz, sadece bir adım ileri gidiyoruz)
        outputs = model(**inputs)
        
        # Son tokenın (next token prediction) logits değerlerini al
        # Shape: (Batch_Size, Vocab_Size) -> (1, 50257)
        next_token_logits = outputs.logits[0, -1, :]
        
        # Sadece ilgilendiğimiz 4 kelimenin puanlarını çekelim
        # target_token_ids = [ID_World, ID_Sports, ID_Business, ID_Sci]
        target_logits = next_token_logits[target_token_ids]
        
        # En yüksek puanı alanın indeksini bul (Argmax)
        # 0: World, 1: Sports, 2: Business, 3: Sci/Tech
        pred_id = torch.argmax(target_logits).item()
    
    predictions.append(pred_id)
    true_labels.append(true_label_id)

# 3. Sonuçlar
acc = accuracy_score(true_labels, predictions)

print(f"\n=========================================")
print(f"BASELINE SKORU:      %85.50")
print(f"GPT-2 (LOGIT):       %{acc * 100:.2f}")
print(f"=========================================")

print("\nFinal Raporu:")
print(classification_report(
    true_labels, 
    predictions, 
    target_names=target_labels,
    zero_division=0
))

--- PROFESYONEL YÖNTEM: LOGIT BAZLI SINIFLANDIRMA ---
Hedef: 'World' -> İzlenen Token ID: 2159 ( World)
Hedef: 'Sports' -> İzlenen Token ID: 7092 ( Sports)
Hedef: 'Business' -> İzlenen Token ID: 7320 ( Business)
Hedef: 'Sci/Tech' -> İzlenen Token ID: 10286 ( Sci)


Olasılıklar Hesaplanıyor: 100%|██████████| 1000/1000 [00:28<00:00, 34.74it/s]


BASELINE SKORU:      %85.50
GPT-2 (LOGIT):       %89.30

Final Raporu:
              precision    recall  f1-score   support

       World       0.96      0.88      0.92       266
      Sports       0.97      0.98      0.97       246
    Business       0.79      0.93      0.85       246
    Sci/Tech       0.88      0.78      0.83       242

    accuracy                           0.89      1000
   macro avg       0.90      0.89      0.89      1000
weighted avg       0.90      0.89      0.89      1000



## Prompt

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Cihaz ayarı
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Çalışma Cihazı: {device}")

model_name = "gpt2"

# 1. Modeli ve Tokenizer'ı Yükle
print("Model yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # GPT-2'de padding yok, EOS kullanıyoruz
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# 2. HAM MODEL TESTİ (Baseline)
# Bakalım model eğitimi olmadan nasıl davranıyor?
input_text = "What is the capital of France?"
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print("\n--- EĞİTİM ÖNCESİ (HAM) ÇIKTI ---")
outputs = model.generate(
    **inputs, 
    max_new_tokens=30, 
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("-----------------------------------")

# 3. ChatML İÇİN ÖZEL TOKENLARIN EKLENMESİ
# Model normalde <|im_start|> ifadesini tanımaz, bunu parçalar.
# Ona bunun "tek bir özel kelime" olduğunu öğretiyoruz.
special_tokens_dict = {
    "additional_special_tokens": ["<|im_start|>", "<|im_end|>"]
}

# Tokenizer'a ekle
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
print(f"\nEklenen Özel Token Sayısı: {num_added_toks}")
print(f"Örnek Token ID (<|im_start|>) : {tokenizer.encode('<|im_start|>')}")

# Modelin Embedding katmanını genişlet (Yeni kelimeler için yer aç)
model.resize_token_embeddings(len(tokenizer))
print(f"Model Embedding Boyutu Güncellendi: {len(tokenizer)}")

Çalışma Cihazı: mps
Model yükleniyor...

--- EĞİTİM ÖNCESİ (HAM) ÇIKTI ---


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


What is the capital of France?

The capital of France is Paris.

The capital of France is Paris.

The capital of France is Paris.

The
-----------------------------------

Eklenen Özel Token Sayısı: 2
Örnek Token ID (<|im_start|>) : [50257]
Model Embedding Boyutu Güncellendi: 50259


In [12]:
from torch.utils.data import Dataset, DataLoader

# 1. Küçük Bir Eğitim Seti (Örnek)
# Modelin farklı tipte sorulara cevap verebildiğini göstermek için çeşitlendirdik.
data = [
    {"instruction": "What is the capital of France?", "solution": "Paris is the capital of France."},
    {"instruction": "Who wrote Romeo and Juliet?", "solution": "William Shakespeare wrote Romeo and Juliet."},
    {"instruction": "What is 2 + 2?", "solution": "The result of 2 + 2 is 4."},
    {"instruction": "Reverse the word 'Hello'.", "solution": "The reverse of 'Hello' is 'olleH'."},
    {"instruction": "What color is the sky?", "solution": "The sky usually appears blue during the day."}
] * 20 # Veriyi çoğaltalım (Toplam 100 örnek) ki epoch başına biraz adım atalım.

# 2. Özel Dataset Sınıfı
class ChatMLDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # ChatML Formatı Oluşturma
        # <|im_start|>system... <|im_start|>user... <|im_start|>assistant...
        
        # A. Prompt Kısmı (Maskelenecek Kısım)
        # Model burayı okuyacak ama burayı tahmin etmeye çalışmayacak.
        prompt_text = (
            f"<|im_start|>system\nYou are a helpful assistant.\n<|im_end|>\n"
            f"<|im_start|>user\n{item['instruction']}\n<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        
        # B. Cevap Kısmı (Öğrenilecek Kısım)
        response_text = f"{item['solution']}\n<|im_end|>"
        
        # C. Tam Metin
        full_text = prompt_text + response_text
        
        # Tokenize Et
        encodings = self.tokenizer(
            full_text, 
            max_length=self.max_length, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        
        input_ids = encodings['input_ids'][0]
        attention_mask = encodings['attention_mask'][0]
        
        # D. Loss Masking (Kritik Nokta)
        labels = input_ids.clone()
        
        # Prompt'un uzunluğunu bulalım (Token sayısı olarak)
        prompt_tokens = self.tokenizer(prompt_text, truncation=True, max_length=self.max_length)["input_ids"]
        prompt_len = len(prompt_tokens)
        
        # 1. Soruyu Maskele (-100 yap)
        labels[:prompt_len] = -100
        
        # 2. Paddingleri Maskele (Boşlukları öğrenmesin)
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# 3. DataLoader Hazırla
train_dataset = ChatMLDataset(data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print(f"Veri Seti Hazır. Toplam Örnek: {len(train_dataset)}")

# 4. Maskeleme Kontrolü (Görsel Test)
sample = train_dataset[0]
print("\n--- MASKELEME KONTROLÜ ---")
print("Input IDs (İlk 20):", sample['input_ids'][:20].tolist())
print("Labels    (İlk 20):", sample['labels'][:20].tolist())
print("\nLabels dizisinde bolca '-100' görmelisiniz. Pozitif sayılar sadece cevabın başladığı yerde olmalı.")

Veri Seti Hazır. Toplam Örnek: 100

--- MASKELEME KONTROLÜ ---
Input IDs (İlk 20): [50257, 10057, 198, 1639, 389, 257, 7613, 8796, 13, 198, 50258, 198, 50257, 7220, 198, 2061, 318, 262, 3139, 286]
Labels    (İlk 20): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Labels dizisinde bolca '-100' görmelisiniz. Pozitif sayılar sadece cevabın başladığı yerde olmalı.


In [13]:
from torch.optim import AdamW
from tqdm.auto import tqdm

# 1. Ayarlar
optimizer = AdamW(model.parameters(), lr=5e-5) # Düşük öğrenme hızı (Hassas ayar)
EPOCHS = 5 # Veri setimiz küçük olduğu için 5 tur döneceğiz

print(f"Eğitim Başlıyor ({EPOCHS} Epoch)...")
model.train() # Modeli eğitim moduna al

# 2. Eğitim Döngüsü
for epoch in range(EPOCHS):
    loop = tqdm(train_loader, leave=True)
    total_loss = 0
    
    for batch in loop:
        # Verileri cihaza taşı (GPU/MPS/CPU)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # A. İleri Yayılım (Forward Pass)
        outputs = model(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels
        )
        
        loss = outputs.loss
        
        # B. Geri Yayılım (Backward Pass)
        optimizer.zero_grad() # Eski gradyanları temizle
        loss.backward()       # Hatayı hesapla
        optimizer.step()      # Ağırlıkları güncelle
        
        # Görselleştirme
        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"--> Epoch {epoch+1} Bitti. Ortalama Kayıp: {avg_loss:.4f}")

print("\nEğitim Tamamlandı! Model artık sohbet etmeyi biliyor.")

Eğitim Başlıyor (5 Epoch)...


Epoch 1: 100%|██████████| 13/13 [00:21<00:00,  1.69s/it, loss=2.88]


--> Epoch 1 Bitti. Ortalama Kayıp: 3.2756


Epoch 2: 100%|██████████| 13/13 [00:02<00:00,  4.73it/s, loss=3.12]


--> Epoch 2 Bitti. Ortalama Kayıp: 2.9284


Epoch 3: 100%|██████████| 13/13 [00:02<00:00,  4.76it/s, loss=2.08]


--> Epoch 3 Bitti. Ortalama Kayıp: 2.4975


Epoch 4: 100%|██████████| 13/13 [00:02<00:00,  4.80it/s, loss=1.85]


--> Epoch 4 Bitti. Ortalama Kayıp: 1.9371


Epoch 5: 100%|██████████| 13/13 [00:02<00:00,  4.75it/s, loss=1.42]

--> Epoch 5 Bitti. Ortalama Kayıp: 1.5795

Eğitim Tamamlandı! Model artık sohbet etmeyi biliyor.


In [14]:
from transformers import StoppingCriteria, StoppingCriteriaList

# 1. Özel Durdurma Sınıfı
class ChatMLStoppingCriteria(StoppingCriteria):
    def __init__(self, stop_token_id):
        self.stop_token_id = stop_token_id

    def __call__(self, input_ids, scores, **kwargs):
        # Eğer üretilen son kelime bizim durma tokenımızsa (<|im_end|>) True döndür
        return input_ids[0, -1] == self.stop_token_id

# 2. Sohbet Fonksiyonu
def generate_chat_response(instruction):
    model.eval() # Değerlendirme modu
    
    # Prompt'u ChatML formatında hazırla (Eğitimdeki gibi)
    # <|im_start|>system... <|im_start|>user... <|im_start|>assistant
    prompt = (
        f"<|im_start|>system\nYou are a helpful assistant.\n<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Durdurma sinyali: <|im_end|> token ID'sini bul
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    stop_criteria = StoppingCriteriaList([ChatMLStoppingCriteria(im_end_id)])
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=50,                  # En fazla 50 kelime konuşsun
            pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=stop_criteria,    # <|im_end|> görünce DUR
            do_sample=False                     # Rastgelelik kapalı (Net cevaplar için)
        )
    
    # Çıktıyı metne çevir
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Cevabı ayıkla (Prompt kısmını at)
    # Metni "<|im_start|>assistant" ifadesinden bölüp sonrasını alıyoruz
    try:
        response = full_text.split("<|im_start|>assistant")[-1]
        # Sondaki <|im_end|> etiketini temizle
        response = response.replace("<|im_end|>", "").strip()
    except:
        response = full_text
        
    return response

# 3. TEST ZAMANI!
print("--- ASİSTAN İLE SOHBET ---")

questions = [
    "What is the capital of France?",
    "What is 2 + 2?",
    "Who wrote Romeo and Juliet?",
    "Reverse the word 'Hello'."
]

for q in questions:
    print(f"Kullanıcı: {q}")
    answer = generate_chat_response(q)
    print(f"Asistan:  {answer}")
    print("-" * 30)

--- ASİSTAN İLE SOHBET ---
Kullanıcı: What is the capital of France?
Asistan:  Paris is the capital of France.
Paris is the capital of France.
Paris is the capital of France.
Paris is the capital of France.
Paris is the capital of France.
Paris is the capital of France.
Paris is
------------------------------
Kullanıcı: What is 2 + 2?
Asistan:  The result of 2 + 2 is 4.
William Shakespeare wrote the play 2 + 2.
William Shakespeare wrote the play 2 + 2.
William Shakespeare wrote the play 2 + 2.
William Shakespeare wrote the play 2 + 2.
------------------------------
Kullanıcı: Who wrote Romeo and Juliet?
Asistan:  William Shakespeare wrote Romeo and Juliet.
William Shakespeare wrote Romeo and Juliet.
William Shakespeare wrote Romeo and Juliet.
William Shakespeare wrote Romeo and Juliet.
William Shakespeare wrote Romeo and Juliet.
William Shakespeare wrote Romeo and Juliet.
William Shakespeare
------------------------------
Kullanıcı: Reverse the word 'Hello'.
Asistan:  The reverse of 'H

In [15]:
from transformers import StoppingCriteria, StoppingCriteriaList

# 1. Özel Durdurma Sınıfı (Aynen kalıyor)
class ChatMLStoppingCriteria(StoppingCriteria):
    def __init__(self, stop_token_id):
        self.stop_token_id = stop_token_id

    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1] == self.stop_token_id

# 2. Sohbet Fonksiyonu (GÜNCELLENDİ)
def generate_chat_response(instruction):
    model.eval()
    
    prompt = (
        f"<|im_start|>system\nYou are a helpful assistant.\n<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    stop_criteria = StoppingCriteriaList([ChatMLStoppingCriteria(im_end_id)])
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=50,
            pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=stop_criteria,
            do_sample=False, 
            
            # --- SİHİRLİ DOKUNUŞ BURADA ---
            repetition_penalty=1.2  # Tekrarı cezalandır (1.0 = ceza yok)
        )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Cevabı ayıklama
    try:
        response = full_text.split("<|im_start|>assistant")[-1]
        response = response.replace("<|im_end|>", "").strip()
        # Eğer hala tekrar varsa (model çok inatçıysa) ilk cümleyi alıp keselim
        if response.count(".") > 1:
             response = response.split(".")[0] + "."
    except:
        response = full_text
        
    return response

# 3. TEST
print("--- DÜZELTİLMİŞ ASİSTAN ÇIKTILARI ---")

questions = [
    "What is the capital of France?",  # Paris...
    "What is 2 + 2?",                  # 4...
    "Who wrote Romeo and Juliet?",     # Shakespeare...
    "Reverse the word 'Hello'."        # olleH...
]

for q in questions:
    ans = generate_chat_response(q)
    print(f"Soru: {q}")
    print(f"Cevap: {ans}")
    print("-" * 30)

--- DÜZELTİLMİŞ ASİSTAN ÇIKTILARI ---
Soru: What is the capital of France?
Cevap: Paris has the capital, Paris as its official name and it's also known for being the home to many famous artists such Assemble de l'Art et des Arts. The city was founded in 1789 by Louis XIV on the island Of Normandy during
------------------------------
Soru: What is 2 + 2?
Cevap: The result of 2 + 4 is 4.
------------------------------
Soru: Who wrote Romeo and Juliet?
Cevap: William Shakespeare wrote Romeo & the Virgin Mary, The Merchant of Venice (1770).
------------------------------
Soru: Reverse the word 'Hello'.
Cevap: The reverse of 'H' is 'olleh', which means 'hello.
------------------------------
